In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# The Usual Suspects

There has been a burglary at a jewelry store.  The [usual suspects](https://en.wikipedia.org/wiki/The_Usual_Suspects) have been arrested.  These are
<ul>
<li>Aaron,</li>
<li>Bernard, and</li>
<li>Caine.</li>
</ul>
Furthermore, the following facts have been established:
<ol>
<li>It is known that at least one of these suspects is indeed guilty.</li>
<li>If Aaron is guilty, he has exactly one accomplice.</li>
<li>If Bernard is innocent, then Caine is inncocent, too.</li>
<li>If exactly two of the suspects are guilty, then Caine is one of them.</li>
<li>If Caine is innocent, then Aaron is guilty.</li>
</ol>
It is our task to identify those suspects that are guilty.

Our first task is to define the set of propositional variables:
$$ \mathcal{P} := \{ \texttt{a}, \texttt{b}, \texttt{c} \} $$
The interpretation is that 
<ul>
<li>$\texttt{a}$ is true iff Aaron is guilty,</li> 
<li>$\texttt{b}$ is true iff Bernard is guilty, and</li>
<li>$\texttt{c}$ is true iff Caine is guilty.  </li>
</ul>

In [2]:
let P = set [ "a"; "b"; "c" ]

Our next task is to translate the facts given above into formulas from propositional logic. 

The statement "It is known that at least one of these suspects is indeed guilty." is translated as follows:
$$ \texttt{a} \vee \texttt{b} \vee \texttt{c}. $$ 

In [3]:
let f1 = "a ∨ b ∨ c"

The statement "If Aaron is guilty, he has exactly one accomplice." is harder to translate into propositional logic. The idea is to split this statement into two statements:
* If Aaron is guilty, he has at least one accomplice.
* If Aaron is guilty, he has at most  one accomplice.

These statements can now be translated into the following formulas:

In [4]:
let f2 = "a → b ∨ c"

In [5]:
let f3 = "a → ¬(b ∧ c)"

The statement "If Bernard is innocent, then Caine is inncocent, too." is a simple implication:

In [6]:
let f4 = "¬b → ¬c"

The statement "If exactly two of the suspects are guilty, then Caine is one of them." is best translated into propositional logic by asking how this statement could be made false.
Obviously, this statement is false if two suspects are guilty, but Caine is innocent.
But this is only possible if Caine is innocent and Aaron and Bernard are guilty.  Hence we can translate this statement as follows:

In [7]:
let f5 = "¬(¬c ∧ a ∧ b)"

The statement "If Caine is innocent, then Aaron is guilty." is an implication:

In [8]:
let f6 = "¬c → a"

We define the set `Fs` of all formulas:

In [9]:
let Fs : Set<string> = set [ f1; f2; f3; f4; f5; f6 ]

We need to transform the strings <tt>f1</tt> to <tt>f6</tt> into values of type `Formula`.  To this end we import the parser for propositional formulas.

In [10]:
importNotebook "Propositional-Logic-Parser.ipynb"

imported 

Propositional-Logic-Parser.ipynb

Next, we transform all formulas into values of type `Formula`.  `Set.map` applies the function `parse` to every element of the set `Fs`.

In [11]:
let Gs : Set<Formula> = Set.map parse Fs
Gs

set [Not (And (And (Not (Var "c"), Var "a"), Var "b")); Or (Or (Var "a", Var "b"), Var "c"); Imp (Var "a", Not (And (Var "b", Var "c"))); Imp (Var "a", Or (Var "b", Var "c")); Imp (Not (Var "b"), Not (Var "c")); Imp (Not (Var "c"), Var "a")]

We are looking for a variable assignment $\mathcal{I}$ that satisfies all formulas in the set <tt>Fs</tt>.  As variable assignments are represented as subsets of the set $\mathcal{P}$ of propositional variables, we can just iterate over all subsets of $\mathcal{P}$.

The function `allSubsets` takes one argument:
- `M` is a finite set.

It returns the list of all subsets of `M`.  The function is *generic*: the type variable `'T` denotes the type of the elements of `M`.

**Implementation:** The idea behind the definition of `allSubsets` is as follows:
1. If `M` is empty, the only subset of `M` is the empty set.
2. Otherwise, let `x` be any element from the set `M`.  We use the smallest element, which is returned by `Set.minElement`.
3. Then there are two kinds of subsets of `M`:
   * Those subsets $A \subseteq M$ that do not contain `x`.
   * Those subsets $B \subseteq M$ that do contain `x`.
4. The list $\mathcal{L}$ of those subsets `A` of `M` that do not contain `x` can be calculated recursively:
   $$ \mathcal{L} = \texttt{allSubsets}(M - \{x\}) $$
5. Adding `x` to the subsets in $\mathcal{L}$ yields all those subsets of $M$ that do contain `x`.  The operator `@` concatenates two lists.

In [12]:
let rec allSubsets (M: Set<'T>) : Set<'T> list =
    if M.IsEmpty then
        [ Set.empty ]
    else
        let x = Set.minElement M
        let L = allSubsets (M.Remove x)
        L @ [ for A in L -> A.Add x ]

In [13]:
allSubsets (set [1; 2; 3])

[set []; set [3]; set [2]; set [2; 3]; set [1]; set [1; 3]; set [1; 2]; set [1; 2; 3]]

The function `evaluate` takes two arguments:
- `f` is a propositional formula,
- `I` is a propositional valuation, represented as the set of those variables that are true.

It returns the value of `f` given the valuation `I`.  We have discussed the details of this function previously.

In [14]:
let rec evaluate (f: Formula) (I: Set<string>) : bool =
    match f with
    | Var p      -> I.Contains p
    | Verum      -> true
    | Falsum     -> false
    | Not g      -> not (evaluate g I)
    | And (g, h) -> evaluate g I && evaluate h I
    | Or  (g, h) -> evaluate g I || evaluate h I
    | Imp (g, h) -> evaluate g I <= evaluate h I
    | Iff (g, h) -> evaluate g I =  evaluate h I
    | Xor (g, h) -> evaluate g I <> evaluate h I

The function `allTrue` takes two arguments:
- `Gs` is a set of propositional formulas,
- `I` is a propositional valuation.

It returns `true` if and only if all formulas from `Gs` are `true` given the valuation `I`.  The function `Set.forall` checks whether a predicate holds for all elements of a set.

In [15]:
let allTrue (Gs: Set<Formula>) (I: Set<string>) : bool =
    Set.forall (fun f -> evaluate f I) Gs

Next, we compute the list of all variable assignments that render all formulas true:

In [16]:
[ for I in allSubsets P do if allTrue Gs I then I ]

[set ["b"; "c"]]

It turns our that there is just one propositional variable assignment that satisfies all formulas from the set <tt>Fs</tt>.  Therefore, the problem has a unique solution: Bernard and Caine are guilty, while Aaron is innocent.